In [ ]:
import main
import prediction
import EDA_PreP
from sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, precision_recall_curve
from sklearn.ensemble import RandomForestClassifier


class training:

    # Logistic LogisticRegression
    model = prediction.getPredictedLogisticRegModel()
    #Predict the response for test dataset
    y_pred = model.predict(X_test)
    main.get_score(y_test, y_pred)
    cm=confusion_matrix(y_test, y_pred)
    print("Confusion matrix \n", cm)

    ## LogisticRegression with grid search START--
    # 1. Setup - GridSearchCV for Best C and Penalty
    # We use 'f1' as the scoring metric instead of accuracy
    param_grid = {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear']
    }

    grid = GridSearchCV(
        LogisticRegression(class_weight='balanced'), 
        param_grid, 
        cv=5, 
        scoring='f1'
    )
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_

    # 2. Threshold Tuning - Moving away from 0.5
    # predict_proba returns [prob_class_0, prob_class_1]
    y_probs = best_model.predict_proba(X_test)[:, 1]

    # Calculate precision-recall pairs for different thresholds
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

    # Calculate F1-score for each threshold to find the optimal point
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    best_threshold = thresholds[np.argmax(f1_scores)]

    # 3. Apply the Custom Threshold
    y_pred_custom = (y_probs >= best_threshold).astype(int)

    print(f"Best C: {grid.best_params_['C']}")
    print(f"Best Threshold: {best_threshold:.4f}")
    print(classification_report(y_test, y_pred_custom))

    ### END

    # Random RandomForestClassifier
    model = prediction.getDecesionTreePredictedModel(entropy,3)

    #Predict the response for test dataset
    y_pred = model.predict(X_test)
    main.get_score(y_test, y_pred)
    cm=confusion_matrix(y_test, y_pred)
    print("Confusion matrix \n", cm)

    ### With Grid Search START ---
        # 1. Define the parameter grid
    # 'balanced' and 'balanced_subsample' are key for imbalanced data
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 20, None],
        'min_samples_leaf': [1, 2, 4],
        'class_weight': ['balanced', 'balanced_subsample']
    }

    # 2. Initialize GridSearchCV
    # scoring='f1' ensures the model optimizes for the minority class
    grid_search = GridSearchCV(
        estimator=RandomForestClassifier(random_state=42),
        param_grid=param_grid,
        cv=5,
        scoring='f1',
        n_jobs=-1,
        verbose=1
    )

    # 3. Fit and predict
    grid_search.fit(X_train, y_train)
    best_rf = grid_search.best_estimator_

    print(f"Best Parameters: {grid_search.best_params_}")
    print(classification_report(y_test, best_rf.predict(X_test)))

    ### END

    #Decision Tree
    model = prediction.getRandomForestPredictedModel(10,42)

    #Predict the response for test dataset
    y_pred = model.predict(X_test)
    main.get_score(y_test, y_pred)
    cm=confusion_matrix(y_test, y_pred)
    print("Confusion matrix \n", cm)

    ### with Grid Search START --
        # 1. Define the parameter grid
    # 'balanced' helps the model learn from minority class samples
    param_grid = {
        'criterion': ['gini', 'entropy'],
        'max_depth': [None, 5, 10, 15, 20],
        'min_samples_leaf': [1, 5, 10, 20],
        'class_weight': ['balanced', None] 
    }

    # 2. Initialize GridSearchCV
    # scoring='f1' is better for imbalanced data than accuracy
    grid_search = GridSearchCV(
        estimator=DecisionTreeClassifier(random_state=42),
        param_grid=param_grid,
        cv=5,
        scoring='f1',
        n_jobs=-1
    )

    # 3. Fit and Evaluate
    grid_search.fit(X_train, y_train)
    best_dt = grid_search.best_estimator_

    print(f"Best Parameters: {grid_search.best_params_}")
    print(classification_report(y_test, best_dt.predict(X_test)))

    ## END

    # XG Boost
    model = prediction.getXgBoostPredictedModel(100,3,0.1)

    #Predict the response for test dataset
    y_pred = model.predict(X_test)
    main.get_score(y_test, y_pred)
    cm=confusion_matrix(y_test, y_pred)
    print("Confusion matrix \n", cm)

    ## with Grid Search--
    # 1. Calculate the initial scale_pos_weight
    # Formula: total_negative_samples / total_positive_samples
    ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)

    # 2. Define the parameter grid
    # We tune scale_pos_weight around the calculated ratio
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1],
        'scale_pos_weight': [1, ratio, ratio * 2] # Test standard vs. theoretical balance
    }

    # 3. Initialize the classifier and grid search
    # Use scoring='f1' or 'roc_auc' to prioritize the minority class
    xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    grid_search = GridSearchCV(
        estimator=xgb_model,
        param_grid=param_grid,
        cv=cv,
        scoring='f1',
        n_jobs=-1
    )

    # 4. Fit and Evaluate
    grid_search.fit(X_train, y_train)
    best_xgb = grid_search.best_estimator_

    print(f"Best Parameters: {grid_search.best_params_}")
    print(classification_report(y_test, best_xgb.predict(X_test)))

    ## END